# JIRA dependency graph

This notebook tries to build a dependency graph for JIRA tickets that form part of an epic.  Given an epic, it:
1. gets the tickets in that epic;
2. obtains the interdependencies of those tickets;
3. constructs a directed graph in which the tickets are the nodes and the edges are the dependencies;
4. adds various attributes to the nodes of the graph;
5. colour-codes the nodes based on the status of the dependencies

### Set up

This section imports packages and sets various configuration parameters.

In [ ]:
import requests

from atlassian import Jira
from pyvis.network import Network

In [ ]:
# provide address of certificate and key, in order to authenticate
CERT = '/etc/pki/tls/certs/client.crt'
KEY = '/etc/pki/tls/private/client.key'

In [ ]:
# provide address of JIRA server (the first part of the URL)
JIRA_SERVER = "https://jira.dev.bbc.co.uk"

In [ ]:
# ticket settings
STORY_POINTS_FIELD = 'customfield_10362'  # the story points appears in a random field name in the Jira response
DEFAULT_STORY_POINTS = 3.0

In [ ]:
# graph settings
NODE_SIZE_SCALE_FACTOR = 5
SAVE_FILE = 'monitoring_ticket_dependencies.html'

## Connect to JIRA

In this section, we connect to Jira.

In [ ]:
session = requests.Session()
session.cert = (CERT, KEY)
session.timeout = 5

In [ ]:
jira = Jira(url=JIRA_SERVER, session=session)

## Get tickets in an epic

Given an epic, denoted by a JIRA number, we want to get all of the tickets in that epic.

In [ ]:
epic = 'DPUB-12646'

In [ ]:
# this returns the tickets in the 'issues' key
tickets = jira.epic_issues(epic)

In [ ]:
tickets['total']

In [ ]:
# the JIRA number that we are familiar with, we need to access the 'key'
tickets = [ticket_data['key'] for ticket_data in tickets['issues']]
print(f"There are {len(tickets)} tickets in the epic {epic}")

## Get ticket data

For each ticket, we want to get the tickets it depends on.  We will save these as a dictionary with keys being the ticket number and the value a list of the tickets it depends on.

We also get the number of story points of a ticket, so that we can vary the node size by the number of story points.

Finally, we also want the status of each ticket, to determine whether a dependent ticket can be worked on or not.

In [ ]:
# instantiate dictionaries to store the data
metadata_dict = {
    'ticket_dependencies': {},
    'ticket_sizes': {},
    'ticket_status': {},
    'ticket_title': {}
}

In [ ]:
for ticket in tickets:
    ticket_details = jira.issue(ticket)
    ticket_size = ticket_details['fields'].get(STORY_POINTS_FIELD, DEFAULT_STORY_POINTS)
    metadata_dict['ticket_status'][ticket] = ticket_details['fields']['status']['name']
    metadata_dict['ticket_title'][ticket] = ticket_details['fields']['summary']
    if not ticket_size:
        ticket_size = DEFAULT_STORY_POINTS
    metadata_dict['ticket_sizes'][ticket] = ticket_size
    dependencies = []
    if linked_issues := ticket_details['fields'].get('issuelinks', None):
        for linked_issue in linked_issues:
            if dependent_on := linked_issue.get('inwardIssue', None):
                dependencies.append(dependent_on['key'])
            else:
                continue
    metadata_dict['ticket_dependencies'][ticket] = dependencies

In [ ]:
dep = set([x for xs in metadata_dict['ticket_dependencies'].values() for x in xs])
len(dep)

In [ ]:
tickets = list(set(tickets).union(dep))
len(tickets)

In [ ]:
for ticket in tickets:
    ticket_details = jira.issue(ticket)
    ticket_size = ticket_details['fields'].get(STORY_POINTS_FIELD, DEFAULT_STORY_POINTS)
    metadata_dict['ticket_status'][ticket] = ticket_details['fields']['status']['name']
    metadata_dict['ticket_title'][ticket] = ticket_details['fields']['summary']
    if not ticket_size:
        ticket_size = DEFAULT_STORY_POINTS
    metadata_dict['ticket_sizes'][ticket] = ticket_size
    dependencies = []
    if linked_issues := ticket_details['fields'].get('issuelinks', None):
        for linked_issue in linked_issues:
            if dependent_on := linked_issue.get('inwardIssue', None):
                dependencies.append(dependent_on['key'])
            else:
                continue
    metadata_dict['ticket_dependencies'][ticket] = dependencies

## Determine node colour

The node colour is determined based on the following rules:
1. it is 'red' if at least one dependency does not have the status 'done';
2. it is 'blue' if it is in progress;
3. it is 'yellow' if all dependencies have the status 'done' and it does not have status 'done' and it is not in progress;
4. it is 'green' if it has status 'done'.

We can apply these in reverse order.

In [ ]:
ticket_colour = {}

In [ ]:
for ticket in tickets:
    # the ticket is done
    if metadata_dict['ticket_status'][ticket] == 'Done':
        ticket_colour[ticket] = 'green'
        continue

    # the ticket is in progress: the status is neither done nor in the backlog
    if metadata_dict['ticket_status'][ticket] != 'Backlog':
        ticket_colour[ticket] = 'blue'
        continue 
        
    # if there are no dependencies, then it can be worked on
    if len(metadata_dict['ticket_dependencies'][ticket]) == 0:
        ticket_colour[ticket] = 'yellow'
        continue

    # get status of all dependencies
    dependencies_status = [metadata_dict['ticket_status'][dependency] for dependency in metadata_dict['ticket_dependencies'][ticket]]

    # all dependencies are completed
    if all(status == 'Done' for status in dependencies_status):
        ticket_colour[ticket] = 'yellow'
        continue

    # if we've go this far, then you can't be worked on
    ticket_colour[ticket] = 'red'

## Generate graph

This section creates a directed graph in which the nodes are the tickets and the edges are the dependencies between them.

In [ ]:
dependency_graph = Network(directed=True, notebook=True)

In [ ]:
# Add the nodes to the graph
for ticket in tickets:
    dependency_graph.add_node(
        ticket, 
        label=ticket,
        title=metadata_dict['ticket_title'][ticket],
        color=ticket_colour[ticket],
        labelHighlightBold=True,
        physics=False,
        shape='dot',
        size=metadata_dict['ticket_sizes'][ticket] * NODE_SIZE_SCALE_FACTOR
    )

In [ ]:
# Add the edges to the graph
for target, sources in metadata_dict['ticket_dependencies'].items():
    if not sources:  # the ticket has no dependencies
        continue
    else:
        for source in sources:
            dependency_graph.add_edge(source=source, to=target, color='black')

In [ ]:
# some visualisation settings
dependency_graph.barnes_hut(overlap=1, gravity=-10000)
dependency_graph.set_edge_smooth('continuous')
dependency_graph.toggle_physics(False)

dependency_graph.show(SAVE_FILE)

## Tickets that can be worked on

Boring!  But here's a list of the tickets that can be worked on.

In [ ]:
ready_tickets = [ticket for ticket, colour in ticket_colour.items() if colour == 'yellow']
print(f"Tickets that can be worked on: {ready_tickets}")